<a href="https://colab.research.google.com/github/670710200/lab-ai-69/blob/master/lab_ai_week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
x = 8.0
lr = 0.1                      # ขนาดก้าว

for i in range(21):
    if i in (0, 1, 2, 3, 5, 10, 20):
        print(f"ก้าว {i:2d}  x = {x:7.4f}  ความสูง = {x*x:8.4f}")
    gradient = 2*x            # ความชันใต้ฝ่าเท้า
    x = x - lr*gradient       # ก้าวสวนทางความชัน

ก้าว  0  x =  8.0000  ความสูง =  64.0000
ก้าว  1  x =  6.4000  ความสูง =  40.9600
ก้าว  2  x =  5.1200  ความสูง =  26.2144
ก้าว  3  x =  4.0960  ความสูง =  16.7772
ก้าว  5  x =  2.6214  ความสูง =   6.8719
ก้าว 10  x =  0.8590  ความสูง =   0.7379
ก้าว 20  x =  0.0922  ความสูง =   0.0085


In [2]:
for lr in [0.01, 0.1, 0.45, 0.55, 1.05]:
    x = 8.0
    for _ in range(20):
        x = x - lr*2*x
    print(f"lr {lr:5.2f}  หลัง 20 ก้าว x = {x:.4f}")

lr  0.01  หลัง 20 ก้าว x = 5.3409
lr  0.10  หลัง 20 ก้าว x = 0.0922
lr  0.45  หลัง 20 ก้าว x = 0.0000
lr  0.55  หลัง 20 ก้าว x = 0.0000
lr  1.05  หลัง 20 ก้าว x = 53.8200


In [3]:
import numpy as np

rng = np.random.default_rng(1)
X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
y = np.array([[0],[1],[1],[0]], float)

In [4]:
X


array([[0., 0.],
       [0., 1.],
       [1., 0.],
       [1., 1.]])

In [5]:
y

array([[0.],
       [1.],
       [1.],
       [0.]])

In [6]:
W1 = rng.normal(0, 1, (2,2)); b1 = np.zeros((1,2))
W2 = rng.normal(0, 1, (2,1)); b2 = np.zeros((1,1))
sig = lambda z: 1/(1+np.exp(-z))

In [7]:
W1

array([[ 0.34558419,  0.82161814],
       [ 0.33043708, -1.30315723]])

In [8]:
W2

array([[0.90535587],
       [0.44637457]])

In [12]:
h = sig(X@W1 + b1)            # ชั้นซ่อนคิด
out = sig(h@W2 + b2)          # ชั้นตอบคิด


In [13]:
print("คำตอบก่อนฝึก", np.round(out.ravel(), 3))
print("loss ก่อนฝึก", round(float(np.mean((out-y)**2)), 4))

คำตอบก่อนฝึก [0.663 0.651 0.699 0.684]
loss ก่อนฝึก 0.2799


In [14]:
lr = 0.5
for epoch in range(20001):
    h = sig(X@W1 + b1)                  # จังหวะ 1 คิดไปข้างหน้า
    out = sig(h@W2 + b2)
    loss = np.mean((out-y)**2)          # จังหวะ 2 วัดความผิด
    if epoch in (0, 100, 1000, 5000, 20000):
        print(f"รอบ {epoch:5d}  loss = {loss:.4f}")

    d_out = (out-y)*out*(1-out)         # จังหวะ 3 ใบตำหนิชั้นตอบ
    d_h = d_out@W2.T * h*(1-h)          # ส่งย้อนไปชั้นซ่อน
    W2 -= lr*h.T@d_out; b2 -= lr*d_out.sum(0)
    W1 -= lr*X.T@d_h;  b1 -= lr*d_h.sum(0)

print("คำตอบหลังฝึก", np.round(out.ravel(), 3))

รอบ     0  loss = 0.2799
รอบ   100  loss = 0.2486
รอบ  1000  loss = 0.0155
รอบ  5000  loss = 0.0007
รอบ 20000  loss = 0.0001
คำตอบหลังฝึก [0.013 0.989 0.989 0.011]


In [16]:
import torch
import torch.nn as nn

In [17]:
torch.manual_seed(1)
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = torch.tensor([[0.],[1.],[1.],[0.]])

In [18]:
model = nn.Sequential(nn.Linear(2,2), nn.Sigmoid(),
                      nn.Linear(2,1), nn.Sigmoid())
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = nn.MSELoss()

In [19]:
for epoch in range(20001):
    out = model(X)                      # คิดไปข้างหน้า
    loss = loss_fn(out, y)              # วัดความผิด
    if epoch in (0, 100, 1000, 5000, 20000):
        print(f"รอบ {epoch:5d}  loss = {loss.item():.4f}")
    opt.zero_grad()
    loss.backward()                     # ใบตำหนิย้อนกลับ ในบรรทัดเดียว
    opt.step()                          # ขยับทุกน้ำหนักหนึ่งก้าว

print("คำตอบหลังฝึก", model(X).detach().numpy().round(3).ravel())

รอบ     0  loss = 0.2548
รอบ   100  loss = 0.2501
รอบ  1000  loss = 0.2500
รอบ  5000  loss = 0.0355
รอบ 20000  loss = 0.0004
คำตอบหลังฝึก [0.021 0.982 0.982 0.019]


In [20]:
def train_xor(lr, epochs=20000, seed=1):
    rng = np.random.default_rng(seed)
    X = np.array([[0,0],[0,1],[1,0],[1,1]], float)
    y = np.array([[0],[1],[1],[0]], float)
    W1 = rng.normal(0,1,(2,2)); b1 = np.zeros((1,2))
    W2 = rng.normal(0,1,(2,1)); b2 = np.zeros((1,1))
    sig = lambda z: 1/(1+np.exp(-z))
    for _ in range(epochs):
        h = sig(X@W1+b1); out = sig(h@W2+b2)
        d_out = (out-y)*out*(1-out)
        d_h = d_out@W2.T * h*(1-h)
        W2 -= lr*h.T@d_out; b2 -= lr*d_out.sum(0)
        W1 -= lr*X.T@d_h;  b1 -= lr*d_h.sum(0)
    h = sig(X@W1+b1); out = sig(h@W2+b2)
    return float(np.mean((out-y)**2)), np.round(out.ravel(), 3)

for lr in [0.01, 0.5, 5.0, 20.0]:
    L, preds = train_xor(lr)
    print(f"lr {lr:5.2f}  loss = {L:.4f}  คำตอบ = {preds}")

lr  0.01  loss = 0.2157  คำตอบ = [0.441 0.404 0.673 0.454]
lr  0.50  loss = 0.0001  คำตอบ = [0.013 0.989 0.989 0.011]
lr  5.00  loss = 0.0000  คำตอบ = [0.004 0.997 0.997 0.003]
lr 20.00  loss = 0.1443  คำตอบ = [0.    0.303 0.998 0.303]
